In [1]:
import numpy as np
import plotly.express as px
import pandas as pd
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

config = util.summary_settings
input_config = util.input_settings

In [2]:
hh = util.get_hh_data()

hh = hh.\
    join(util.get_parcel_landuse_data(), how="left",left_on='hhno',right_on='parcelid').\
    join(util.get_parcel_geog(), how="left",left_on='hhparcel',right_on='ParcelID').\
        to_pandas()

person = util.get_person_data().to_pandas()
tour = util.get_tour_data().to_pandas()


In [3]:
df_tour = tour.copy()
df_hh = hh.copy()
df_person = person.copy()

# auto availability
df_hh['auto_count_driver'] = df_hh['hhvehs']-df_hh['drivers']
df_hh['auto_available_driver'] = np.where(df_hh['drivers']<=0, "no driver",
                                          np.where(df_hh['hhvehs']<=0, "no car",
                                                   np.where(df_hh['auto_count_driver']<0, "cars fewer than drivers", "enough cars")))

df_person = df_person.merge(df_hh, how='left', on=['hhno','source']) # get auto ownership from hh data
df_tour = df_tour.merge(df_person, how='left', on=['pno','hhno','source'])

In [4]:
es_tour = df_tour.loc[(df_tour['parent']==0) & (df_tour['pdpurp']==3)].copy()

In [5]:
df_plot = es_tour.groupby(['source','tmodetp_label'])['toexpfac'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['toexpfac']. \
    apply(lambda x: x / float(x.sum()))

df_plot_ct = es_tour.groupby(['source','tmodetp_label'])['toexpfac'].count().reset_index(). \
    rename(columns={'toexpfac':'sample count'})
df_plot = df_plot.merge(df_plot_ct, on=['source','tmodetp_label'])

fig = px.bar(df_plot.sort_values(by=['source']), x="tmodetp_label", y="percentage", color="source",
             barmode="group",hover_data=['sample count'],title="escort tour mode")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  xaxis = dict(dtick = 1, categoryorder='category ascending'),
                  yaxis=dict(tickformat=".2%"))
fig.show()

### mode choice by segment

In [6]:
def plot_mode_choice(df: pd.DataFrame, grp_var: str, order_list: dict, title_name: str, n_nol: int, height=400, width=800):
    df_plot = df.groupby(['source',grp_var,'tmodetp_label'])['toexpfac'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source',grp_var], group_keys=False)['toexpfac']. \
        apply(lambda x: x / float(x.sum()))

    df_plot_ct = df.groupby(['source',grp_var,'tmodetp_label'])['toexpfac'].count().reset_index(). \
        rename(columns={'toexpfac':'sample count'})
    df_plot = df_plot.merge(df_plot_ct, on=['source',grp_var,'tmodetp_label'])

    fig = px.bar(df_plot.sort_values(['source','tmodetp_label']),
                 x="percentage", y="tmodetp_label", color="source",barmode="group",
                 facet_col=grp_var, facet_col_wrap=n_nol, orientation='h',
                 category_orders=order_list,hover_data=['sample count'],
                 title="escort tour mode choice by " + title_name)
    fig.update_layout(height=height, width=width)
    fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
    fig.for_each_xaxis(lambda a: a.update(tickformat = ".1%"))
    fig.show()

In [7]:
incl_county = ["King", "Kitsap", "Pierce", "Snohomish"]

plot_mode_choice(es_tour.loc[es_tour["CountyName"].isin(incl_county)],"CountyName",
                 {"CountyName":incl_county,
                  "tmodetp_label": util.tmodetp_cat.values()},
                 "home county",
                 2,600)



In [8]:
plot_mode_choice(es_tour,"pptyp_label",
                 {"pptyp_label":util.pptyp_cat.values(),
                  "tmodetp_label": util.tmodetp_cat.values()},
                 "person type",
                 2,1000)

In [9]:
plot_mode_choice(es_tour,"hhsize_4+",
                 {"hhsize_4+":["1","2","3","4+"],
                  "tmodetp_label":util.tmodetp_cat.values()},
                 "household size",2,600)

In [10]:
plot_mode_choice(es_tour.loc[es_tour['hhvehs_4+']!="-1"],
                 "hhvehs_4+",
                 {"hhvehs_4+":["0","1","2","3","4+"],
                  "tmodetp_label":util.tmodetp_cat.values()},
                 "auto ownership",2,800)

In [11]:
plot_mode_choice(es_tour,
                 "hhcu5",
                 {"tmodetp_label":util.tmodetp_cat.values()},
                 "HH with Children Under 5",2,800)

In [12]:
plot_mode_choice(es_tour,
                 "hh515",
                 {"tmodetp_label":util.tmodetp_cat.values()},
                 "HH with Children 5-15",2,800)

In [13]:
plot_mode_choice(es_tour,
                 "hhhsc",
                 {
                  "tmodetp_label":util.tmodetp_cat.values()},
                 "HH with Driving Age Students",2,800)

In [14]:
plot_mode_choice(es_tour.loc[es_tour['auto_available_driver'] != "no dirver"], "auto_available_driver",
                 {"auto_available_driver": ["no car", "cars fewer than drivers", "enough cars"],
                  "tmodetp_label":util.tmodetp_cat.values()},
                 "auto availability (driver, showing only households with at least one driver)", 3, 500, 900)